============================================================
# SGTA: Synalytic Graph Transformer Augmentation

Full version with direct influence on memory

============================================================


In [ ]:
from __future__ import annotations

import argparse
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    brier_score_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.mixture import GaussianMixture
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.stats import kurtosis
from torch.optim import AdamW
from torch.utils.data import DataLoader

try:
    from datasets import Dataset, DatasetDict, load_dataset
    from transformers import (
        AutoConfig,
        AutoModel,
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        get_cosine_schedule_with_warmup,
    )
except Exception as exc:  # pragma: no cover
    raise RuntimeError(
        "This framework requires `datasets` and `transformers`. "
        "Install them before running."
    ) from exc

2026-04-18 00:57:25.215443: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-18 00:57:25.215492: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-18 00:57:25.215523: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-18 00:57:25.223083: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-18 00:57:26.054122: W tensorflow/compiler/

In [ ]:
@dataclass
class Paths:
    root: Path = Path("runs")
    artifacts: Path = Path("runs/artifacts")
    logs: Path = Path("runs/logs")
    checkpoints: Path = Path("runs/checkpoints")
    figures: Path = Path("runs/figures")

    def ensure(self) -> None:
        for p in [self.root, self.artifacts, self.logs, self.checkpoints, self.figures]:
            p.mkdir(parents=True, exist_ok=True)


@dataclass
class DatasetConfig:
    dataset_name: str = "glue"
    dataset_subset: Optional[str] = "sst2"
    text_column: str = "sentence"
    label_column: str = "label"
    train_split: str = "train"
    val_split: str = "validation"
    test_split: Optional[str] = "test"
    max_length: int = 128
    num_labels: int = 2


@dataclass
class TrainConfig:
    model_name: str = "bert-base-uncased"
    batch_size: int = 32
    eval_batch_size: int = 64
    epochs: int = 3
    lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    eval_every: int = 100
    target_val_acc: float = 0.90
    max_steps: Optional[int] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


@dataclass
class ConceptConfig:
    proj_dim: int = 7
    num_caps: int = 128
    beta: float = 10.0
    lam: float = 0.6
    a0: float = 1e-2
    alpha: float = 2e-3
    ema_eta: float = 0.1
    amax_frac: float = 0.25
    kinetics_every: int = 50
    graph_tau: float = 0.1
    graph_rho: float = 0.1
    activation_tau: float = 0.10
    overlap_theta: float = 0.0
    ndc_threshold: float = 1e-3


@dataclass
class ExperimentConfig:
    seeds: List[int] = field(default_factory=lambda: [0, 1, 2, 3, 4])
    worst_seed_scan_seeds: List[int] = field(default_factory=lambda: [0, 1, 2, 3, 4])
    run_multiseed: bool = True
    save_hidden_states: bool = True
    hidden_layer: int = -1
    concept_kmeans_k: int = 32
    concept_gmm_k: int = 32
    sphere_threshold_quantile: float = 0.90
    ood_dataset_name: Optional[str] = None
    ood_dataset_subset: Optional[str] = None
    ood_text_column: Optional[str] = None
    ood_label_column: Optional[str] = None


@dataclass
class FullConfig:
    paths: Paths = field(default_factory=Paths)
    dataset: DatasetConfig = field(default_factory=DatasetConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    concept: ConceptConfig = field(default_factory=ConceptConfig)
    experiment: ExperimentConfig = field(default_factory=ExperimentConfig)


# ============================================================
# Metrics
# ============================================================


def ece_score(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        acc = (predictions[mask] == labels[mask]).mean()
        conf = confidences[mask].mean()
        ece += mask.mean() * abs(acc - conf)
    return float(ece)



def nll_score(probs: np.ndarray, labels: np.ndarray, eps: float = 1e-12) -> float:
    clipped = np.clip(probs, eps, 1.0)
    return float(-np.log(clipped[np.arange(len(labels)), labels]).mean())



def multiclass_brier_score(probs: np.ndarray, labels: np.ndarray, num_labels: int) -> float:
    onehot = np.eye(num_labels)[labels]
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))



def steps_to_target(values: Sequence[float], eval_every: int, target: float) -> Optional[int]:
    for i, v in enumerate(values, start=1):
        if v >= target:
            return i * eval_every
    return None



def pairwise_angular_distance(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    x = x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)
    y = y / np.clip(np.linalg.norm(y, axis=1, keepdims=True), 1e-12, None)
    cos = np.clip(x @ y.T, -1.0, 1.0)
    return np.arccos(cos)



def spectral_gap_lambda2_from_adj(adj: np.ndarray, normalized: bool = True) -> float:
    w = np.array(adj, dtype=np.float64)
    w = 0.5 * (w + w.T)
    np.fill_diagonal(w, 0.0)
    d = w.sum(axis=1)
    eps = 1e-12
    if normalized:
        d_inv_sqrt = np.diag(1.0 / np.sqrt(d + eps))
        lap = np.eye(w.shape[0]) - d_inv_sqrt @ w @ d_inv_sqrt
    else:
        lap = np.diag(d) - w
    evals = np.linalg.eigvalsh(lap)
    evals.sort()
    if len(evals) < 2:
        return 0.0
    return float(evals[1])


# ============================================================
# Data
# ============================================================


class TextDatasetBuilder:
    def __init__(self, cfg: FullConfig):
        self.cfg = cfg
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.train.model_name)
        self.collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

    def load(self) -> DatasetDict:
        from datasets import load_dataset
        return load_dataset(
            "csv",
            data_files={
                "train": "data/local_sst2/train.csv",
                "validation": "data/local_sst2/validation.csv",
            }
        )


    def tokenize_dataset(self, ds: DatasetDict) -> DatasetDict:
        text_col = self.cfg.dataset.text_column
        label_col = self.cfg.dataset.label_column
        max_length = self.cfg.dataset.max_length

        def tok(batch: Dict[str, Any]) -> Dict[str, Any]:
            out = self.tokenizer(
                batch[text_col],
                truncation=True,
                max_length=max_length,
            )
            out["labels"] = batch[label_col]
            return out

        ds_tok = ds.map(tok, batched=True)
        keep = {"input_ids", "attention_mask", "labels", text_col, label_col}
        for split in list(ds_tok.keys()):
            remove = [c for c in ds_tok[split].column_names if c not in keep]
            ds_tok[split] = ds_tok[split].remove_columns(remove)
            ds_tok[split].set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
        return ds_tok

    def loaders(self, ds_tok: DatasetDict) -> Tuple[DataLoader, DataLoader, Optional[DataLoader]]:
        train_loader = DataLoader(
            ds_tok[self.cfg.dataset.train_split],
            batch_size=self.cfg.train.batch_size,
            shuffle=True,
            collate_fn=self.collator,
        )
        val_loader = DataLoader(
            ds_tok[self.cfg.dataset.val_split],
            batch_size=self.cfg.train.eval_batch_size,
            shuffle=False,
            collate_fn=self.collator,
        )
        test_loader = None
        if self.cfg.dataset.test_split and self.cfg.dataset.test_split in ds_tok:
            test_loader = DataLoader(
                ds_tok[self.cfg.dataset.test_split],
                batch_size=self.cfg.train.eval_batch_size,
                shuffle=False,
                collate_fn=self.collator,
            )
        return train_loader, val_loader, test_loader


# ============================================================
# SGTA modules
# ============================================================



def l2_normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return x / (x.norm(dim=-1, keepdim=True) + eps)


class CapProjector(nn.Module):
    def __init__(self, hidden_size: int, d: int):
        super().__init__()
        self.proj = nn.Linear(hidden_size, d, bias=False)
        nn.init.orthogonal_(self.proj.weight)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return l2_normalize(self.proj(h))


class CapMemory(nn.Module):
    def __init__(self, cfg: ConceptConfig):
        super().__init__()
        self.cfg = cfg
        c = torch.randn(cfg.num_caps, cfg.proj_dim)
        self.register_buffer("centers", l2_normalize(c))
        self.register_buffer("areas", torch.full((cfg.num_caps,), cfg.a0))
        self.register_buffer("hit_count", torch.zeros(cfg.num_caps))
        self.register_buffer("z_accum", torch.zeros(cfg.num_caps, cfg.proj_dim))

    @staticmethod
    def sphere_area(d: int) -> torch.Tensor:
        return 2.0 * torch.pi ** (d / 2) / torch.exp(torch.lgamma(torch.tensor(d / 2.0)))

    @staticmethod
    def area_to_cos_r(a: torch.Tensor, d: int) -> torch.Tensor:
        s = CapMemory.sphere_area(d).to(a.device)
        c_d = 0.5 * s
        cos_r = 1.0 - a / c_d
        return cos_r.clamp(-0.9999, 0.9999)

    def forward(self, z: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        sim = torch.einsum("btd,md->btm", z, self.centers)
        cos_r = self.area_to_cos_r(self.areas, self.cfg.proj_dim)
        s = sim - cos_r.view(1, 1, -1)
        h_act = F.softplus(self.cfg.beta * s)
        bias = self.cfg.lam * h_act.max(dim=-1).values
        with torch.no_grad():
            hit = (s > 0).float()
            self.hit_count += hit.sum(dim=(0, 1))
            self.z_accum += torch.einsum("btd,btm->md", z, hit)
        return bias, h_act

    @torch.no_grad()
    def step_kinetics(self) -> None:
        mask = self.hit_count > 0
        if mask.any():
            z_bar = torch.zeros_like(self.centers)
            z_bar[mask] = self.z_accum[mask] / self.hit_count[mask].unsqueeze(-1)
            new_c = l2_normalize((1.0 - self.cfg.ema_eta) * self.centers + self.cfg.ema_eta * z_bar)
            self.centers.copy_(new_c)
        s = self.sphere_area(self.cfg.proj_dim).to(self.areas.device)
        amax = self.cfg.amax_frac * s
        grow = self.cfg.alpha * torch.exp(-self.areas / amax)
        self.areas.copy_(torch.minimum(0.7 * self.areas + 0.3 * self.cfg.a0 + grow, torch.full_like(self.areas, amax)))
        self.hit_count.zero_()
        self.z_accum.zero_()


class ConceptGraph(nn.Module):
    def __init__(self, cfg: ConceptConfig):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("W", torch.tensor([]))
        self.register_buffer("coact", torch.tensor([]))

    @torch.no_grad()
    def update(self, centers: torch.Tensor, areas: torch.Tensor, h_act: torch.Tensor) -> None:
        m = centers.shape[0]
        cos_r = CapMemory.area_to_cos_r(areas, centers.shape[-1])
        r = torch.arccos(cos_r).unsqueeze(0)
        cos_sum = torch.cos(r.T + r)
        cos_cc = centers @ centers.T
        w_geo = torch.sigmoid((cos_cc - cos_sum) / self.cfg.graph_tau)
        h = h_act.mean(dim=(0, 1))
        h_n = h / (h.max() + 1e-8)
        w_co = torch.outer(h_n, h_n)
        w = (w_geo + w_co).clamp(0.0, 1.0)
        if self.W.numel() == 0:
            self.W = w.clone()
            self.coact = w_co.clone()
        else:
            self.W = (1.0 - self.cfg.graph_rho) * self.W + self.cfg.graph_rho * w
            self.coact = (1.0 - self.cfg.graph_rho) * self.coact + self.cfg.graph_rho * w_co

    def laplacian_gap(self) -> Optional[float]:
        if self.W.numel() == 0:
            return None
        return spectral_gap_lambda2_from_adj(self.W.detach().cpu().numpy(), normalized=True)


class SGTABiasMode:
    NONE = "none"
    LOGIT = "logit"
    ATTN = "attn"
    DUAL = "dual"


class SGTABertClassifier(nn.Module):
    """
    A compact SGTA wrapper around a Hugging Face sequence classifier.

    Supports three modes:
    - none: baseline
    - logit: classifier-level bias
    - attn: attention-mask bias before self-attention
    - dual: both
    """

    def __init__(self, model_name: str, num_labels: int, concept_cfg: ConceptConfig, mode: str):
        super().__init__()
        self.mode = mode
        config = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
        config.output_attentions = True
        config.output_hidden_states = True
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
        hidden_size = self.model.config.hidden_size
        self.projector = CapProjector(hidden_size, concept_cfg.proj_dim)
        self.memory = CapMemory(concept_cfg)
        self.graph = ConceptGraph(concept_cfg)
        self.concept_cfg = concept_cfg
        self._hooks: List[Any] = []
        self._last_h_act: Optional[torch.Tensor] = None
        self._steps = 0
        if mode in {SGTABiasMode.ATTN, SGTABiasMode.DUAL}:
            self._install_attention_hooks()

    def _install_attention_hooks(self) -> None:
        for name, module in self.model.named_modules():
            if name.endswith("attention"):
                self._hooks.append(module.register_forward_pre_hook(self._save_hidden_pre, with_kwargs=True))
        for name, module in self.model.named_modules():
            if name.endswith("attention.self"):
                self._hooks.append(module.register_forward_pre_hook(self._attn_pre_hook, with_kwargs=True))

    def _save_hidden_pre(self, module: nn.Module, args: Tuple[Any, ...], kwargs: Dict[str, Any]):
        hs = args[0] if len(args) > 0 else kwargs.get("hidden_states", None)
        if hs is not None and hasattr(module, "self"):
            module.self._sgta_hidden = hs
        return args, kwargs

    def _attn_pre_hook(self, module: nn.Module, args: Tuple[Any, ...], kwargs: Dict[str, Any]):
        hidden_states = getattr(module, "_sgta_hidden", None)
        if hidden_states is None:
            return args, kwargs
        with torch.no_grad():
            z = self.projector(hidden_states)
            b, h_act = self.memory(z)
            self._last_h_act = h_act.detach()
            bias_mask = b.unsqueeze(1).unsqueeze(1)
            am = kwargs.get("attention_mask", None)
            if am is None:
                am = torch.zeros_like(bias_mask, dtype=hidden_states.dtype, device=hidden_states.device)
            kwargs["attention_mask"] = am + bias_mask.to(am.dtype)
        return args, kwargs

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: Optional[torch.Tensor] = None):
        out = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        if self.mode not in {SGTABiasMode.LOGIT, SGTABiasMode.DUAL}:
            return out

        hidden = out.hidden_states[-1]
        z = self.projector(hidden)
        b, h_act = self.memory(z)
        self._last_h_act = h_act
        cls_bias = b[:, 0]
        logits = out.logits.clone()
        if logits.shape[-1] == 2:
            logits[:, 1] = logits[:, 1] + cls_bias
        else:
            logits = logits + cls_bias.unsqueeze(-1)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        out.logits = logits
        out.loss = loss if loss is not None else out.loss
        return out

    @torch.no_grad()
    def sgta_step(self) -> None:
        self._steps += 1
        if self._steps % self.concept_cfg.kinetics_every == 0:
            self.memory.step_kinetics()
            if self._last_h_act is not None:
                self.graph.update(self.memory.centers, self.memory.areas, self._last_h_act)

    def remove_hooks(self) -> None:
        for h in self._hooks:
            h.remove()
        self._hooks.clear()


# ============================================================
# Training / inference
# ============================================================


@dataclass
class EvalOutput:
    logits: np.ndarray
    labels: np.ndarray
    probs: np.ndarray
    preds: np.ndarray
    accuracy: float
    ece: float
    nll: float
    brier: float


@dataclass
class TrainOutput:
    history: Dict[str, List[float]]
    best_val_acc: float
    best_step: int
    steps_to_target: Optional[int]
    checkpoint_path: Optional[str]


class Trainer:
    def __init__(self, cfg: FullConfig):
        self.cfg = cfg

    def build_model(self, mode: str) -> SGTABertClassifier:
        return SGTABertClassifier(
            model_name=self.cfg.train.model_name,
            num_labels=self.cfg.dataset.num_labels,
            concept_cfg=self.cfg.concept,
            mode=mode,
        ).to(self.cfg.train.device)

    @torch.no_grad()
    def evaluate(self, model: nn.Module, loader: DataLoader) -> EvalOutput:
        model.eval()
        logits_all: List[torch.Tensor] = []
        labels_all: List[torch.Tensor] = []
        for batch in loader:
            batch = {k: v.to(self.cfg.train.device) for k, v in batch.items()}
            out = model(**batch)
            logits_all.append(out.logits.detach().cpu())
            labels_all.append(batch["labels"].detach().cpu())
        logits = torch.cat(logits_all).numpy()
        labels = torch.cat(labels_all).numpy()
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
        preds = probs.argmax(axis=1)
        return EvalOutput(
            logits=logits,
            labels=labels,
            probs=probs,
            preds=preds,
            accuracy=float(accuracy_score(labels, preds)),
            ece=ece_score(probs, labels),
            nll=nll_score(probs, labels),
            brier=multiclass_brier_score(probs, labels, self.cfg.dataset.num_labels),
        )

    def fit(self, model: SGTABertClassifier, train_loader: DataLoader, val_loader: DataLoader, run_name: str) -> TrainOutput:
        num_steps = self.cfg.train.epochs * math.ceil(len(train_loader))
        if self.cfg.train.max_steps is not None:
            num_steps = min(num_steps, self.cfg.train.max_steps)
        optim = AdamW(model.parameters(), lr=self.cfg.train.lr, weight_decay=self.cfg.train.weight_decay)
        sched = get_cosine_schedule_with_warmup(
            optim,
            int(num_steps * self.cfg.train.warmup_ratio),
            num_steps,
        )
        history: Dict[str, List[float]] = {
            "train_loss": [],
            "val_acc": [],
            "ece": [],
            "nll": [],
            "brier": [],
            "lambda2": [],
            "overlap": [],
            "ndc": [],
        }
        best_val_acc = -1.0
        best_step = 0
        steps_seen = 0
        best_path = self.cfg.paths.checkpoints / f"{run_name}.pt"

        for epoch in range(self.cfg.train.epochs):
            model.train()
            for batch in train_loader:
                steps_seen += 1
                batch = {k: v.to(self.cfg.train.device) for k, v in batch.items()}
                out = model(**batch)
                loss = out.loss
                if loss is None:
                    raise RuntimeError("Model returned no training loss.")
                loss.backward()
                optim.step()
                sched.step()
                optim.zero_grad(set_to_none=True)
                model.sgta_step()
                history["train_loss"].append(float(loss.detach().cpu().item()))

                if steps_seen % self.cfg.train.eval_every == 0:
                    ev = self.evaluate(model, val_loader)
                    history["val_acc"].append(ev.accuracy)
                    history["ece"].append(ev.ece)
                    history["nll"].append(ev.nll)
                    history["brier"].append(ev.brier)
                    lambda2 = model.graph.laplacian_gap() or 0.0
                    history["lambda2"].append(lambda2)
                    overlap = calc_overlap(
                        model.memory.centers.detach(),
                        model.memory.areas.detach(),
                        theta=self.cfg.concept.overlap_theta,
                    )
                    history["overlap"].append(overlap)
                    ndc = ndc_count(model._last_h_act, eps=self.cfg.concept.ndc_threshold)
                    history["ndc"].append(float(ndc or 0))
                    if ev.accuracy > best_val_acc:
                        best_val_acc = ev.accuracy
                        best_step = steps_seen
                        # torch.save(model.state_dict(), best_path) SMOKE TEST

                if steps_seen >= num_steps:
                    break
            if steps_seen >= num_steps:
                break

        return TrainOutput(
            history=history,
            best_val_acc=best_val_acc,
            best_step=best_step,
            steps_to_target=steps_to_target(history["val_acc"], self.cfg.train.eval_every, self.cfg.train.target_val_acc),
            # checkpoint_path=str(best_path) if best_path.exists() else None, SMOKE TEST
            checkpoint_path=None,
        )


# ============================================================
# Hidden-state extraction and concept fitting
# ============================================================


@dataclass
class HiddenStateBundle:
    states: np.ndarray
    labels: np.ndarray
    logits: np.ndarray
    probs: np.ndarray
    texts: Optional[List[str]] = None


class HiddenStateExtractor:
    def __init__(self, cfg: FullConfig):
        self.cfg = cfg
        self.backbone = AutoModel.from_pretrained(cfg.train.model_name, output_hidden_states=True).to(cfg.train.device)

    @torch.no_grad()
    def extract(self, loader: DataLoader, classifier: Optional[nn.Module] = None) -> HiddenStateBundle:
        self.backbone.eval()
        if classifier is not None:
            classifier.eval()
        states_list: List[np.ndarray] = []
        labels_list: List[np.ndarray] = []
        logits_list: List[np.ndarray] = []
        probs_list: List[np.ndarray] = []

        for batch in loader:
            batch_device = {k: v.to(self.cfg.train.device) for k, v in batch.items()}
            out = self.backbone(
                input_ids=batch_device["input_ids"],
                attention_mask=batch_device["attention_mask"],
            )
            h = out.hidden_states[self.cfg.experiment.hidden_layer][:, 0, :]
            states_list.append(h.detach().cpu().numpy())
            labels_list.append(batch_device["labels"].detach().cpu().numpy())
            if classifier is not None:
                clf_out = classifier(**batch_device)
                logits = clf_out.logits.detach().cpu().numpy()
            else:
                logits = np.zeros((h.shape[0], self.cfg.dataset.num_labels), dtype=np.float32)
            probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
            logits_list.append(logits)
            probs_list.append(probs)

        states = np.concatenate(states_list, axis=0)
        labels = np.concatenate(labels_list, axis=0)
        logits = np.concatenate(logits_list, axis=0)
        probs = np.concatenate(probs_list, axis=0)
        return HiddenStateBundle(states=states, labels=labels, logits=logits, probs=probs)


@dataclass
class ConceptFit:
    centers: np.ndarray
    radii: np.ndarray
    assignments: np.ndarray
    normalized_states: np.ndarray


class SphericalConceptFitter:
    def __init__(self, cfg: FullConfig):
        self.cfg = cfg

    def fit_kmeans_caps(self, states: np.ndarray, k: Optional[int] = None) -> ConceptFit:
        if k is None:
            k = self.cfg.experiment.concept_kmeans_k

        n = len(states)
        if n == 0:
            raise ValueError("No states provided for concept fitting.")
        k = min(k, n)

        z = states / np.clip(np.linalg.norm(states, axis=1, keepdims=True), 1e-12, None)
        km = KMeans(n_clusters=k, random_state=0, n_init=10)
        assignments = km.fit_predict(z)
        raw_centers = km.cluster_centers_
        centers = raw_centers / np.clip(np.linalg.norm(raw_centers, axis=1, keepdims=True), 1e-12, None)
        d = pairwise_angular_distance(z, centers)
        radii = np.zeros(k, dtype=np.float64)
        q = self.cfg.experiment.sphere_threshold_quantile
        for i in range(k):
            mask = assignments == i
            if not mask.any():
                radii[i] = 0.0
                continue
            radii[i] = float(np.quantile(d[mask, i], q))
        return ConceptFit(centers=centers, radii=radii, assignments=assignments, normalized_states=z)

    def fit_gmm(self, states: np.ndarray, k: Optional[int] = None) -> Dict[str, Any]:
        if k is None:
            k = self.cfg.experiment.concept_gmm_k

        n = len(states)
        if n == 0:
            raise ValueError("No states provided for GMM fitting.")
        k = min(k, n)

        z = states / np.clip(np.linalg.norm(states, axis=1, keepdims=True), 1e-12, None)
        gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=0)
        assignments = gmm.fit_predict(z)
        return {
            "means": gmm.means_,
            "covariances": gmm.covariances_,
            "assignments": assignments,
            "normalized_states": z,
        }


# ============================================================
# Core geometry / graph analyses
# ============================================================



def calc_overlap(centers: torch.Tensor, areas: torch.Tensor, theta: float = 0.0) -> float:
    m, d = centers.shape
    cos_r = CapMemory.area_to_cos_r(areas, d)
    r = torch.arccos(cos_r)
    cos_sum = torch.cos(r.view(-1, 1) + r.view(1, -1))
    cos_cc = (centers @ centers.T).clamp(-1, 1)
    mask = torch.ones(m, m, device=centers.device, dtype=torch.bool).triu(1)
    valid = cos_cc[mask] >= (cos_sum[mask] - theta)
    return float(valid.float().mean().item())



def ndc_count(h_act: Optional[torch.Tensor], k: int = 1, eps: float = 1e-3) -> Optional[int]:
    if h_act is None:
        return None
    hits = (h_act > eps).float().sum(dim=(0, 1))
    return int((hits >= k).sum().item())



def center_drift(prev_centers: np.ndarray, cur_centers: np.ndarray) -> float:
    return float(np.linalg.norm(cur_centers - prev_centers, axis=1).mean())



def fit_sphericality_statistics(fit: ConceptFit) -> Dict[str, float]:
    z = fit.normalized_states
    k = fit.centers.shape[0]
    within_vars: List[float] = []
    within_kurt: List[float] = []
    for i in range(k):
        mask = fit.assignments == i
        if mask.sum() < 5:
            continue
        d = pairwise_angular_distance(z[mask], fit.centers[i : i + 1]).reshape(-1)
        within_vars.append(float(np.var(d)))
        within_kurt.append(float(kurtosis(d, fisher=False, bias=False)))
    return {
        "mean_within_angular_var": float(np.mean(within_vars)) if within_vars else float("nan"),
        "mean_within_distance_kurtosis": float(np.mean(within_kurt)) if within_kurt else float("nan"),
    }



def fit_graph_from_concepts(fit: ConceptFit) -> np.ndarray:
    centers = fit.centers
    radii = fit.radii
    cos_cc = np.clip(centers @ centers.T, -1.0, 1.0)
    rsum = radii[:, None] + radii[None, :]
    geo = 1.0 / (1.0 + np.exp(-(cos_cc - np.cos(rsum)) / 0.1))
    np.fill_diagonal(geo, 0.0)
    return geo



def match_concepts_hungarian(c1: np.ndarray, c2: np.ndarray) -> Dict[str, Any]:
    d = pairwise_angular_distance(c1, c2)
    row, col = linear_sum_assignment(d)
    return {
        "rows": row.tolist(),
        "cols": col.tolist(),
        "mean_match_angle": float(d[row, col].mean()),
        "max_match_angle": float(d[row, col].max()),
    }


# ============================================================
# Experiment suite
# ============================================================


class ExperimentSuite:
    def __init__(self, cfg: FullConfig):
        self.cfg = cfg
        self.paths = cfg.paths
        self.paths.ensure()
        self.builder = TextDatasetBuilder(cfg)
        self.trainer = Trainer(cfg)
        self.extractor = HiddenStateExtractor(cfg)
        self.fitter = SphericalConceptFitter(cfg)

    def load_data(self) -> Tuple[DatasetDict, DataLoader, DataLoader, Optional[DataLoader]]:
        ds = self.builder.load()
        ds_tok = self.builder.tokenize_dataset(ds)
        train_loader, val_loader, test_loader = self.builder.loaders(ds_tok)
        return ds_tok, train_loader, val_loader, test_loader


    @staticmethod
    def to_jsonable(obj: Any) -> Any:
        if isinstance(obj, dict):
            return {str(k): ExperimentSuite.to_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [ExperimentSuite.to_jsonable(v) for v in obj]
        if isinstance(obj, tuple):
            return [ExperimentSuite.to_jsonable(v) for v in obj]
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            val = float(obj)
            if math.isnan(val) or math.isinf(val):
                return None
            return val
        if isinstance(obj, float):
            if math.isnan(obj) or math.isinf(obj):
                return None
            return obj
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, Path):
            return str(obj)
        return obj

    def save_json(self, name: str, payload: Dict[str, Any]) -> None:
        out = self.paths.logs / f"{name}.json"
        with open(out, "w", encoding="utf-8") as f:
            json.dump(self.to_jsonable(payload), f, ensure_ascii=False, indent=2)

    def run_single_training(self, seed: int, mode: str, tag: str) -> Dict[str, Any]:
        set_seed(seed)
        _, train_loader, val_loader, test_loader = self.load_data()
        model = self.trainer.build_model(mode)
        run_name = f"{tag}_{mode}_seed{seed}"
        train_out = self.trainer.fit(model, train_loader, val_loader, run_name)
        if train_out.checkpoint_path:
            model.load_state_dict(torch.load(train_out.checkpoint_path, map_location=self.cfg.train.device))
        val_eval = self.trainer.evaluate(model, val_loader)
        test_eval = self.trainer.evaluate(model, test_loader) if test_loader is not None else None
        return {
            "seed": seed,
            "mode": mode,
            "tag": tag,
            "train": {
                "best_val_acc": train_out.best_val_acc,
                "best_step": train_out.best_step,
                "steps_to_target": train_out.steps_to_target,
                "history": train_out.history,
            },
            "val": asdict(val_eval),
            "test": asdict(test_eval) if test_eval else None,
        }

    # --------------------------------------------------------
    # MAP 1. Geometry: sphere validity and alternatives
    # --------------------------------------------------------
    def experiment_geometry_map(self, seed: int = 0) -> Dict[str, Any]:
        res = self.run_single_training(seed=seed, mode=SGTABiasMode.NONE, tag="geometry")
        _, _, val_loader, _ = self.load_data()
        classifier = self.trainer.build_model(SGTABiasMode.NONE)
        if res["train"]["history"]:
            ckpt = self.paths.checkpoints / f"geometry_none_seed{seed}.pt"
            if ckpt.exists():
                classifier.load_state_dict(torch.load(ckpt, map_location=self.cfg.train.device))
        bundle = self.extractor.extract(val_loader, classifier=classifier)

        sphere_fit = self.fitter.fit_kmeans_caps(bundle.states)
        sphere_stats = fit_sphericality_statistics(sphere_fit)

        gmm_fit = self.fitter.fit_gmm(bundle.states)
        pca = PCA(n_components=min(16, bundle.states.shape[1]))
        pca.fit(bundle.states)
        explained = float(np.sum(pca.explained_variance_ratio_[: min(8, len(pca.explained_variance_ratio_))]))

        payload = {
            "seed": seed,
            "sphere_stats": sphere_stats,
            "sphere_num_concepts": int(sphere_fit.centers.shape[0]),
            "sphere_mean_radius": float(np.mean(sphere_fit.radii)),
            "gmm_mean_trace_cov": float(np.mean([np.trace(c) for c in gmm_fit["covariances"]])),
            "pca_top8_explained_variance": explained,
        }
        self.save_json("geometry_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 2. Dynamics: birth / death / drift / overlap
    # --------------------------------------------------------
    def experiment_dynamics_map(self, seed: int = 0) -> Dict[str, Any]:
        res = self.run_single_training(seed=seed, mode=SGTABiasMode.DUAL, tag="dynamics")
        hist = res["train"]["history"]
        val_acc = hist["val_acc"]
        payload = {
            "seed": seed,
            "num_eval_points": len(val_acc),
            "val_acc_curve": val_acc,
            "lambda2_curve": hist["lambda2"],
            "overlap_curve": hist["overlap"],
            "ndc_curve": hist["ndc"],
            "ece_curve": hist["ece"],
            "best_val_acc": res["train"]["best_val_acc"],
            "steps_to_target": res["train"]["steps_to_target"],
        }
        self.save_json("dynamics_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 3. Functionality: baseline vs SGTA modes
    # --------------------------------------------------------
    def experiment_functional_map(self, seeds: Optional[List[int]] = None) -> Dict[str, Any]:
        if seeds is None:
            seeds = self.cfg.experiment.seeds
        modes = [SGTABiasMode.NONE, SGTABiasMode.LOGIT, SGTABiasMode.ATTN, SGTABiasMode.DUAL]
        all_runs: List[Dict[str, Any]] = []
        for seed in seeds:
            for mode in modes:
                all_runs.append(self.run_single_training(seed=seed, mode=mode, tag="functional"))

        summary: Dict[str, Dict[str, float]] = {}
        for mode in modes:
            rows = [r for r in all_runs if r["mode"] == mode]
            summary[mode] = {
                "mean_best_val_acc": float(np.mean([r["train"]["best_val_acc"] for r in rows])),
                "std_best_val_acc": float(np.std([r["train"]["best_val_acc"] for r in rows])),
                "mean_val_ece": float(np.mean([r["val"]["ece"] for r in rows])),
                "mean_val_nll": float(np.mean([r["val"]["nll"] for r in rows])),
                "mean_steps_to_target": float(np.mean([r["train"]["steps_to_target"] or np.nan for r in rows])),
            }
        payload = {"runs": all_runs, "summary": summary}
        self.save_json("functional_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 4. Graph: structure and semantic organization
    # --------------------------------------------------------
    def experiment_graph_map(self, seed: int = 0) -> Dict[str, Any]:
        self.run_single_training(seed=seed, mode=SGTABiasMode.DUAL, tag="graph")
        _, _, val_loader, _ = self.load_data()
        classifier = self.trainer.build_model(SGTABiasMode.DUAL)
        ckpt = self.paths.checkpoints / f"graph_dual_seed{seed}.pt"
        if ckpt.exists():
            classifier.load_state_dict(torch.load(ckpt, map_location=self.cfg.train.device))
        bundle = self.extractor.extract(val_loader, classifier=classifier)
        fit = self.fitter.fit_kmeans_caps(bundle.states)
        graph = fit_graph_from_concepts(fit)
        gap = spectral_gap_lambda2_from_adj(graph, normalized=True)
        density = float((graph > 0.5).mean())
        payload = {
            "seed": seed,
            "num_nodes": int(graph.shape[0]),
            "lambda2": gap,
            "density_threshold_0_5": density,
            "mean_radius": float(np.mean(fit.radii)),
        }
        self.save_json("graph_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 5. Stability: multi-seed alignment
    # --------------------------------------------------------
    def experiment_stability_map(self, seeds: Optional[List[int]] = None) -> Dict[str, Any]:
        if seeds is None:
            seeds = self.cfg.experiment.seeds
        concept_sets: List[np.ndarray] = []
        per_seed: List[Dict[str, Any]] = []

        for seed in seeds:
            self.run_single_training(seed=seed, mode=SGTABiasMode.DUAL, tag="stability")
            _, _, val_loader, _ = self.load_data()
            classifier = self.trainer.build_model(SGTABiasMode.DUAL)
            ckpt = self.paths.checkpoints / f"stability_dual_seed{seed}.pt"
            if ckpt.exists():
                classifier.load_state_dict(torch.load(ckpt, map_location=self.cfg.train.device))
            bundle = self.extractor.extract(val_loader, classifier=classifier)
            fit = self.fitter.fit_kmeans_caps(bundle.states)
            concept_sets.append(fit.centers)
            per_seed.append({"seed": seed, "mean_radius": float(np.mean(fit.radii))})

        matches: List[Dict[str, Any]] = []
        base = concept_sets[0]
        for i in range(1, len(concept_sets)):
            match = match_concepts_hungarian(base, concept_sets[i])
            match["seed_ref"] = seeds[0]
            match["seed_cmp"] = seeds[i]
            matches.append(match)

        payload = {
            "per_seed": per_seed,
            "matches_vs_seed0": matches,
            "mean_match_angle": float(np.mean([m["mean_match_angle"] for m in matches])) if matches else float("nan"),
        }
        self.save_json("stability_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 6. OOD and error prediction
    # --------------------------------------------------------
    def experiment_ood_map(self, seed: int = 0) -> Dict[str, Any]:
        self.run_single_training(seed=seed, mode=SGTABiasMode.DUAL, tag="ood")
        _, _, val_loader, _ = self.load_data()
        classifier = self.trainer.build_model(SGTABiasMode.DUAL)
        ckpt = self.paths.checkpoints / f"ood_dual_seed{seed}.pt"
        if ckpt.exists():
            classifier.load_state_dict(torch.load(ckpt, map_location=self.cfg.train.device))
        bundle = self.extractor.extract(val_loader, classifier=classifier)
        fit = self.fitter.fit_kmeans_caps(bundle.states)
        d = pairwise_angular_distance(fit.normalized_states, fit.centers)
        min_d = d.min(axis=1)
        errors = (bundle.probs.argmax(axis=1) != bundle.labels).astype(np.int64)
        roc = float(roc_auc_score(errors, min_d)) if len(np.unique(errors)) > 1 else float("nan")
        payload = {
            "seed": seed,
            "error_rate": float(errors.mean()),
            "ood_proxy_auc": roc,
            "mean_min_distance": float(min_d.mean()),
            "mean_min_distance_on_errors": float(min_d[errors == 1].mean()) if errors.sum() > 0 else float("nan"),
            "mean_min_distance_on_correct": float(min_d[errors == 0].mean()) if (errors == 0).sum() > 0 else float("nan"),
        }
        self.save_json("ood_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 7. Ablations
    # --------------------------------------------------------
    def experiment_ablation_map(self, seed: int = 0) -> Dict[str, Any]:
        results = {
            "baseline": self.run_single_training(seed, SGTABiasMode.NONE, "ablation"),
            "logit_only": self.run_single_training(seed, SGTABiasMode.LOGIT, "ablation"),
            "attn_only": self.run_single_training(seed, SGTABiasMode.ATTN, "ablation"),
            "dual": self.run_single_training(seed, SGTABiasMode.DUAL, "ablation"),
        }
        payload = {
            k: {
                "best_val_acc": v["train"]["best_val_acc"],
                "ece": v["val"]["ece"],
                "steps_to_target": v["train"]["steps_to_target"],
            }
            for k, v in results.items()
        }
        self.save_json("ablation_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 8. Worst-seed-first diagnostic
    # --------------------------------------------------------
    def experiment_worst_seed_map(self) -> Dict[str, Any]:
        baseline_runs = [
            self.run_single_training(seed=s, mode=SGTABiasMode.NONE, tag="worstseed_scan")
            for s in self.cfg.experiment.worst_seed_scan_seeds
        ]
        baseline_runs = sorted(
            baseline_runs,
            key=lambda x: (x["train"]["best_val_acc"], x["train"]["steps_to_target"] or 10 ** 9),
        )
        worst = baseline_runs[0]
        worst_seed = worst["seed"]
        candidates = [
            self.run_single_training(worst_seed, SGTABiasMode.LOGIT, "worstseed"),
            self.run_single_training(worst_seed, SGTABiasMode.ATTN, "worstseed"),
            self.run_single_training(worst_seed, SGTABiasMode.DUAL, "worstseed"),
        ]
        payload = {
            "worst_seed": worst_seed,
            "baseline": worst,
            "variants": candidates,
        }
        self.save_json("worst_seed_map", payload)
        return payload

    # --------------------------------------------------------
    # MAP 9. Attribute-alignment placeholder
    # --------------------------------------------------------
    def experiment_attribute_alignment_map(
        self,
        states: np.ndarray,
        binary_attributes: np.ndarray,
    ) -> Dict[str, Any]:
        """
        Plug-in experiment for CUB / VAW / custom attribute labels.
        states: [N, H]
        binary_attributes: [N, A]
        """
        fit = self.fitter.fit_kmeans_caps(states)
        d = pairwise_angular_distance(fit.normalized_states, fit.centers)
        scores = -d
        aucs: List[float] = []
        precs: List[float] = []
        recs: List[float] = []
        for a in range(binary_attributes.shape[1]):
            y = binary_attributes[:, a]
            if len(np.unique(y)) < 2:
                continue
            clf = LogisticRegression(max_iter=500)
            clf.fit(scores, y)
            p = clf.predict_proba(scores)[:, 1]
            aucs.append(float(roc_auc_score(y, p)))
            yh = (p >= 0.5).astype(int)
            precs.append(float(precision_score(y, yh, zero_division=0)))
            recs.append(float(recall_score(y, yh, zero_division=0)))
        payload = {
            "mean_auc": float(np.mean(aucs)) if aucs else float("nan"),
            "mean_precision": float(np.mean(precs)) if precs else float("nan"),
            "mean_recall": float(np.mean(recs)) if recs else float("nan"),
            "num_attributes_evaluated": len(aucs),
        }
        self.save_json("attribute_alignment_map", payload)
        return payload

    # --------------------------------------------------------
    # Full run
    # --------------------------------------------------------
    def run_all_maps(self) -> Dict[str, Any]:
        results = {
            "geometry": self.experiment_geometry_map(seed=self.cfg.experiment.seeds[0]),
            "dynamics": self.experiment_dynamics_map(seed=self.cfg.experiment.seeds[0]),
            "functional": self.experiment_functional_map(),
            "graph": self.experiment_graph_map(seed=self.cfg.experiment.seeds[0]),
            "stability": self.experiment_stability_map(),
            "ood": self.experiment_ood_map(seed=self.cfg.experiment.seeds[0]),
            "ablation": self.experiment_ablation_map(seed=self.cfg.experiment.seeds[0]),
            "worst_seed": self.experiment_worst_seed_map(),
        }
        self.save_json("all_maps_summary", results)
        return results


# ============================================================
# CLI
# ============================================================



def build_default_config() -> FullConfig:
    cfg = FullConfig()
    cfg.paths.ensure()
    return cfg



def main() -> None:
    parser = argparse.ArgumentParser(description="SGTA experiment framework")
    parser.add_argument(
        "--experiment",
        type=str,
        default="all",
        choices=[
            "all",
            "geometry",
            "dynamics",
            "functional",
            "graph",
            "stability",
            "ood",
            "ablation",
            "worst_seed",
        ],
    )
    args, _ = parser.parse_known_args()

    cfg = build_default_config()
    suite = ExperimentSuite(cfg)

    if args.experiment == "all":
        suite.run_all_maps()
    elif args.experiment == "geometry":
        suite.experiment_geometry_map(seed=cfg.experiment.seeds[0])
    elif args.experiment == "dynamics":
        suite.experiment_dynamics_map(seed=cfg.experiment.seeds[0])
    elif args.experiment == "functional":
        suite.experiment_functional_map()
    elif args.experiment == "graph":
        suite.experiment_graph_map(seed=cfg.experiment.seeds[0])
    elif args.experiment == "stability":
        suite.experiment_stability_map()
    elif args.experiment == "ood":
        suite.experiment_ood_map(seed=cfg.experiment.seeds[0])
    elif args.experiment == "ablation":
        suite.experiment_ablation_map(seed=cfg.experiment.seeds[0])
    elif args.experiment == "worst_seed":
        suite.experiment_worst_seed_map()


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For deterministic behaviour (may slow down training)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# import pandas as pd
# import os

# os.makedirs("data/sst2", exist_ok=True)

# # train
# !wget https://dl.fbaipublicfiles.com/glue/data/SST-2/train.tsv -O data/sst2/train.tsv
# #
# # dev
# !wget https://dl.fbaipublicfiles.com/glue/data/SST-2/dev.tsv -O data/sst2/dev.tsv

# import pandas as pd
# import os

# os.makedirs("data/local_sst2", exist_ok=True)

# train_df = pd.DataFrame({
#     "sentence": [
#         "this movie is wonderful",
#         "i loved this film",
#         "amazing story",
#         "great acting",
#         "bad movie",
#         "terrible plot",
#         "awful acting",
#         "boring film",
#     ],
#     "label": [1, 1, 1, 1, 0, 0, 0, 0]
# })

# val_df = pd.DataFrame({
#     "sentence": [
#         "excellent film",
#         "very bad movie"
#     ],
#     "label": [1, 0]
# })

# train_df.to_csv("data/local_sst2/train.csv", index=False)
# val_df.to_csv("data/local_sst2/validation.csv", index=False)


smoke test

In [ ]:
cfg = build_default_config()
cfg.train.eval_every = 1
cfg.experiment.seeds = [0]
cfg.experiment.worst_seed_scan_seeds = [0]
cfg.train.epochs = 1
cfg.train.batch_size = 16
cfg.train.eval_batch_size = 32
cfg.dataset.test_split = None
cfg.experiment.concept_kmeans_k = 2
cfg.experiment.concept_gmm_k = 2

suite = ExperimentSuite(cfg)
result = suite.experiment_functional_map(seeds=[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
result["summary"]

{'none': {'mean_best_val_acc': 0.5,
  'std_best_val_acc': 0.0,
  'mean_val_ece': 0.052759408950805664,
  'mean_val_nll': 0.6893584728240967,
  'mean_steps_to_target': nan},
 'logit': {'mean_best_val_acc': 0.5,
  'std_best_val_acc': 0.0,
  'mean_val_ece': 0.049706459045410156,
  'mean_val_nll': 0.6881810426712036,
  'mean_steps_to_target': nan},
 'attn': {'mean_best_val_acc': 0.5,
  'std_best_val_acc': 0.0,
  'mean_val_ece': 0.05335986614227295,
  'mean_val_nll': 0.6891769170761108,
  'mean_steps_to_target': nan},
 'dual': {'mean_best_val_acc': 0.5,
  'std_best_val_acc': 0.0,
  'mean_val_ece': 0.050042569637298584,
  'mean_val_nll': 0.6869672536849976,
  'mean_steps_to_target': nan}}

 yay!


The smoke test completed successfully.

Next steps require:
- computational resources for full-scale verification,
- a production version of the pipeline (not smoke, but still
  computationally reasonable).

<!-- #yay!The smoke test completed successfully.
Next steps require:
- computational resources for full-scale verification,
- a production version of the pipeline (not smoke, but still
  computationally reasonable). -->

In [ ]:
# ============================================================
# Production run (pending compute resources)
# ============================================================
# This cell runs the full experimental pipeline on real SST-2.
# Requires: GPU with ≥16GB VRAM, ~2-3 hours for all 9 maps.
# Not intended for smoke testing — uncomment when ready.


In [ ]:
# cfg = build_default_config()

# # --- Scaling: more data, more steps, but still reasonable ---

cfg.dataset.dataset_name = "glue"
cfg.dataset.dataset_subset = "sst2"
cfg.dataset.max_length = 128
cfg.dataset.num_labels = 2

cfg.train.model_name = "bert-base-uncased"
cfg.train.batch_size = 32
cfg.train.eval_batch_size = 64
cfg.train.epochs = 3
cfg.train.lr = 3e-5
cfg.train.weight_decay = 0.01
cfg.train.warmup_ratio = 0.06
cfg.train.eval_every = 100
cfg.train.target_val_acc = 0.90

cfg.concept.proj_dim = 7
cfg.concept.num_caps = 128
cfg.concept.beta = 10.0
cfg.concept.lam = 0.6
cfg.concept.a0 = 1e-2
cfg.concept.alpha = 2e-3
cfg.concept.ema_eta = 0.1
cfg.concept.amax_frac = 0.25
cfg.concept.kinetics_every = 50
cfg.concept.graph_tau = 0.1
cfg.concept.graph_rho = 0.1
cfg.concept.activation_tau = 0.10
cfg.concept.overlap_theta = 0.0
cfg.concept.ndc_threshold = 1e-3

cfg.experiment.seeds = [0, 1, 2, 3, 4]
cfg.experiment.worst_seed_scan_seeds = [0, 1, 2, 3, 4]
cfg.experiment.run_multiseed = True
cfg.experiment.save_hidden_states = True
cfg.experiment.hidden_layer = -1
cfg.experiment.concept_kmeans_k = 32
cfg.experiment.concept_gmm_k = 32
cfg.experiment.sphere_threshold_quantile = 0.90

In [ ]:
# # --- Run ---
suite = ExperimentSuite(cfg)
results = suite.run_all_maps()
print("Production run complete. Results saved to runs/logs/")